# Stage 2 Notebook 13 - Exp2H CLRKD lane head + ASL classification rescue

Exp2G (NB12) broke the geometry plateau: val/lane_point_mae dropped from 0.4011 (Exp2E e10 plateau) to 0.3244 with monotonic decrease, and val/matched_line_iou rose from ~0.08 to 0.4285. The ROI gather + multi-scale bilinear sampling fix worked.

Exp2G also exposed three new problems:

1. **Lane existence classification stuck at the focal-uniform-predictor equilibrium.** val/lane/cls_pos and cls_neg both froze at ~0.07-0.09, pos_score - neg_score collapsed from 0.04 at epoch 1 to 0.004 at epoch 10, and pred_lanes ~= 192 * batch (every prior predicted positive). Standard focal loss has near-flat gradients at p=0.5 which let the geometry loss dominate while cls flatlined.
2. **Detection mAP50 stuck at ~0.003.** Pre-existing problem unrelated to the lane head; investigated separately.
3. **GCA gates frozen at 0.5** and **lambda_lane pinned at the floor 0.2**. Combined with backbone_lr_mult=0.1 the backbone barely updated.

Exp2H targets problems 1 and 3:

- **Asymmetric Focal Loss (ASL)** with gamma_pos=0, gamma_neg=4 instead of standard focal. Removes the flat-gradient zone for positives and amplifies the negative push.
- **Bump w_cls 2.0 -> 6.0** and reduce w_iou 2.0 -> 1.0. Geometry already works at iou=2.0; give cls more relative weight now.
- **Concat the 3-d prior embedding into cls_head input** (`cls_uses_prior_embedding: true`). Geometry losses train per_lane to be position-invariant once a prior is matched; appending the prior position gives cls a feature it can use to discriminate.
- **Raise lambda_min 0.2 -> 0.5** so the backbone receives more lane gradient now that geometry is healthy.

Backbone (RMT + GCA), detection head (DETR), CLRKDLaneHead (3 stages, 36 sample points, dynamic-k matching) all unchanged from Exp2G.

Reference: Ben-Baruch et al. 2020 "Asymmetric Loss For Multi-Label Classification".

### Run mode

1. Keep `DEBUG_MODE = True` for the first run.
2. After the smoke and debug run succeed, change to `False` for the 10-epoch short run.
3. Output is mirrored to the notebook cell, the Colab runtime log, and a Drive log file.
4. The dataset tar is read from Drive; do not rerun Notebook 00.

In [1]:
import os, sys, subprocess, textwrap
from google.colab import drive
os.environ['PYTHONUNBUFFERED'] = '1'
drive.mount('/content/drive')

REPO_ROOT = '/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane'
if not os.path.isdir(REPO_ROOT):
    raise FileNotFoundError(f'Missing project root: {REPO_ROOT}')
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pyyaml', 'scipy', 'opencv-python-headless', 'tqdm', 'matplotlib'])
print('repo:', REPO_ROOT)

from stage2.scripts.notebook_utils import run_streaming
LOG_DIR = '/content/drive/MyDrive/EcoCAR/training_runs/notebook_logs'
os.makedirs(LOG_DIR, exist_ok=True)

Mounted at /content/drive
repo: /content/drive/MyDrive/EcoCAR/yolop_vehicle_lane


In [2]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp08_rmt_gca_clrkd_asl_cls_rescue_joint.yaml'
LOG_FILE = os.path.join(LOG_DIR, f'{Path(CONFIG).stem}_smoke.log')

# Smoke test: tiny forward + backward through ASL + prior-embedding cls path.
# Must print 'OK exp08_*.yaml' with shapes before training is attempted.
run_streaming([sys.executable, '-u', 'stage2/scripts/smoke_test_joint_models.py', CONFIG], log_path=LOG_FILE)

[run_streaming] command: /usr/bin/python3 -u stage2/scripts/smoke_test_joint_models.py stage2/configs/exp08_rmt_gca_clrkd_asl_cls_rescue_joint.yaml
[run_streaming] log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/exp08_rmt_gca_clrkd_asl_cls_rescue_joint_smoke.log
OK exp08_rmt_gca_clrkd_asl_cls_rescue_joint.yaml
  lane_shape=(1, 16, 72, 2) det_shape=(1, 4, 4)
  lane_loss=4.0231 det_loss=3.5993 grad_cos=-0.1281 lambda_lane=0.0500
  gate_stats={'gate/det_mean': 0.49958452582359314, 'gate/lane_mean': 0.4985475242137909, 'gate/det_sat_low': 0.0, 'gate/det_sat_high': 0.0, 'gate/lane_sat_low': 0.0, 'gate/lane_sat_high': 0.0}
[run_streaming] return_code=0


0

In [3]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp08_rmt_gca_clrkd_asl_cls_rescue_joint.yaml'
CURVE_TAR = '/content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar'
CURVE_ROOT = '/content/bdd100k_clrkd_curve'

DEBUG_MODE = False

if DEBUG_MODE:
    RUN_TAG = 'debug'
    EPOCHS = 2
    BATCH_SIZE = 4
    LIMIT_TRAIN = 512
    LIMIT_VAL = 256
    PRINT_EVERY = 5
else:
    RUN_TAG = 'short10'
    EPOCHS = 10
    BATCH_SIZE = 8
    LIMIT_TRAIN = 3000
    LIMIT_VAL = 1000
    PRINT_EVERY = 5

run_stem = Path(CONFIG).stem + '_' + RUN_TAG
WORK_DIR = f'/content/{run_stem}'
OUTPUT_TAR = f'/content/drive/MyDrive/EcoCAR/training_runs/{run_stem}.tar'
LOG_FILE = os.path.join(LOG_DIR, f'{run_stem}_train.log')

cmd = [
    sys.executable, '-u', 'stage2/scripts/train_joint_model_experiment.py',
    '--config', CONFIG,
    '--curve-tar', CURVE_TAR,
    '--curve-root', CURVE_ROOT,
    '--work-dir', WORK_DIR,
    '--output-tar', OUTPUT_TAR,
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--limit-train', str(LIMIT_TRAIN),
    '--limit-val', str(LIMIT_VAL),
    '--force-extract',
    '--print-every', str(PRINT_EVERY),
]

print('DEBUG_MODE:', DEBUG_MODE, flush=True)
print('About to run:', ' '.join(cmd), flush=True)
print('Output tar:', OUTPUT_TAR, flush=True)
print('Visible log file:', LOG_FILE, flush=True)
run_streaming(cmd, log_path=LOG_FILE)

DEBUG_MODE: False
About to run: /usr/bin/python3 -u stage2/scripts/train_joint_model_experiment.py --config stage2/configs/exp08_rmt_gca_clrkd_asl_cls_rescue_joint.yaml --curve-tar /content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar --curve-root /content/bdd100k_clrkd_curve --work-dir /content/exp08_rmt_gca_clrkd_asl_cls_rescue_joint_short10 --output-tar /content/drive/MyDrive/EcoCAR/training_runs/exp08_rmt_gca_clrkd_asl_cls_rescue_joint_short10.tar --epochs 10 --batch-size 8 --limit-train 3000 --limit-val 1000 --force-extract --print-every 5
Output tar: /content/drive/MyDrive/EcoCAR/training_runs/exp08_rmt_gca_clrkd_asl_cls_rescue_joint_short10.tar
Visible log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/exp08_rmt_gca_clrkd_asl_cls_rescue_joint_short10_train.log
[run_streaming] command: /usr/bin/python3 -u stage2/scripts/train_joint_model_experiment.py --config stage2/configs/exp08_rmt_gca_clrkd_asl_cls_rescue_joint.yaml --curve-tar /content/drive/MyDrive

0

## What to watch in Exp2H training

Reference Exp2G epoch 10: val_lane_point_mae=0.3244, val_matched_line_iou=0.4285, val_lane_exist_best_f1=0.075, pos-neg_score_diff=0.004.

Strong signals that the cls rescue worked:

- `val/lane_exist_best_f1` rises to >= 0.40 by epoch 5 and >= 0.65 by epoch 10. (Exp2G e10 was 0.075.)
- `val/lane_exist_pos_score_mean - val/lane_exist_neg_score_mean` >= 0.15 by epoch 10. (Exp2G e10 was 0.004; we want at least 30x more.)
- `val/lane/cls_pos` and `val/lane/cls_neg` separate. Currently both at ~0.07-0.09; pos should drop substantially below neg.
- `val/lane_point_mae` does NOT regress (must remain <= 0.34). The cls bump should not hurt geometry.
- `val/matched_line_iou` does NOT drop below 0.30. Same constraint.
- `pred_lanes / batch` drops from ~192 (Exp2G all-positive) toward ~5-10 (real predictions).
- `train/mtl/lambda_lane_runtime` should now sit higher than Exp2G's pinned 0.2 because lambda_min was raised to 0.5.

Failure signals that mean we need a different rescue:

- `val/lane_exist_best_f1` still below 0.20 at epoch 10 -> ASL is not enough; consider hard-negative mining (top-k hardest unmatched priors), or add a separate lane existence supervision via the auxiliary mask logits.
- Geometry regresses (point_mae > 0.36) -> w_cls=6 / w_iou=1 weight shift was too aggressive; pull it back to 4 / 1.5.
- pred_lanes drops to ~0 with recall collapse -> ASL gamma_neg=4 is too aggressive; reduce to 2 or 3.

After short10 finishes, run Notebook 08 (it now includes exp08 candidates) to plot per-epoch trends and compare Exp2G vs Exp2H side-by-side.